# Electricity GHG emission factors — unified pipeline

Each section parses one database, calls `db.calc_ghg(inplace=True)` (added in MARIO `dev_ghg`), reshapes the GHG row into long-form CSV via `common.emission_factors`, and exports it to `export/`.
Run `merge.py` afterwards to build `export/combined_results.csv` in `kg/EUR`.

In [ ]:
import warnings
import pandas as pd
import mario

from common import load_config, db_path, emission_factors, export_efs, should_skip
from database_properties import properties

mario.set_log_verbosity('info')
warnings.filterwarnings('ignore')

# How to react when a target CSV is already in export/:
#   'ask'       -> prompt y/N for each file
#   'skip'      -> always skip
#   'overwrite' -> always recompute
OVERWRITE_POLICY = 'ask'

cfg, base = load_config()
{
    'legacy_shared_root': str(base) if base is not None else None,
    'configured_databases': list(cfg['databases']),
}

## EXIOBASE monetary IOT (ixi / pxp)

In [ ]:
p = properties['exiobase_monetary_iot']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in p['years']:
        for system in p['systems']:
            archive = f'IOT_{year}_{system}'
            folder = db_path(cfg, base, 'EXIOBASE',
                             version=version, table=table,
                             system=system, year=year, archive=archive)
            print(f'\n=== {name} {version} {table} {system} {year} ===')

            if should_skip(name, table, version, year, system,
                           policy=OVERWRITE_POLICY):
                continue

            if not folder.exists():
                folder.parent.mkdir(parents=True, exist_ok=True)
                try:
                    mario.download_exiobase3(
                        path=str(folder.parent), years=year,
                        system=system, table=table, version=version,
                    )
                except Exception as exc:
                    print(f'[skip] download failed: {exc}')
                    continue

            try:
                db = mario.parse_exiobase(
                    path=str(folder), table=table,
                    unit='Monetary',
                )
            except Exception as exc:
                print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
                continue

            db.calc_ghg(profile='exiobase_monetary', inplace=True)
            efs = emission_factors(db, 'Sector', p['labels_list'][system])
            export_efs(efs, name, table, version, year, system)

## EORA26

In [ ]:
p = properties['eora26']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in p['years']:
        path = db_path(cfg, base, 'EORA26', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_eora(path=str(path), multi_region=True)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db.calc_ghg(profile='eora', inplace=True)
        efs = emission_factors(db, 'Sector', p['labels_list'],
                               ghg_unit='Gg CO2eq')
        export_efs(efs, name, table, version, year)

## EMERGING

In [ ]:
mario.set_compute_method('inverse')

p = properties['emerging']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in p['years']:
        path = db_path(cfg, base, 'EMERGING', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_emerging(path=str(path), year=year)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db.calc_ghg(profile='emerging', inplace=True)
        efs = emission_factors(db, 'Sector', p['labels_list'],
                               ghg_unit='Mt CO2eq')
        export_efs(efs, name, table, version, year)

## GLORIA (Activity + Commodity, merged into one Sector column)

In [ ]:
p = properties['gloria']
name, table = p['name'], p['table']

for version in p['versions']:
    for year in p['years']:
        path = db_path(cfg, base, 'GLORIA', year=year)
        print(f'\n=== {name} {version} {table} {year} ===')

        if should_skip(name, table, version, year, policy=OVERWRITE_POLICY):
            continue

        try:
            db = mario.parse_gloria(path=str(path), year=year)
        except Exception as exc:
            print(f'[skip] parse failed: {type(exc).__name__}: {exc}')
            continue

        db.calc_ghg(profile='gloria', inplace=True)
        efs = pd.concat([
            emission_factors(db, lvl, labels).rename(columns={lvl: 'Sector'})
            for lvl, labels in p['labels_list'].items()
        ], axis=0, ignore_index=True)
        export_efs(efs, name, table, version, year)

## Merge all CSVs and convert to kg/EUR

Equivalent to `python merge.py`.

In [ ]:
%run merge.py